### System prompt로 AI에게 역할 주기
- 같은 LLM이라도 어떤 역할을 주느냐에 따라 비서가 될 수도 선생님이 될 수도 있음

1. 고객 상담용 챗봇
2. 번역 잘하는 챗봇
3. 나를 위로해주는 챗봇
4. 잔소리? 다그치는 챗봇

#### 다양한 상황과 역할 부여해서 챗봇 만들어보기!
- 대화내용 유지,
- stream 등 기능도 구현해보기
- system prompt 내가 원하는 스타일대로 대화가 이뤄지도록 하는 것이 핵심


- 출력 형식도 정해보기
- 예)
    - 메뉴 1
    - 메뉴 2 .. 이런 식이나
- 금지사항, 제약 사항도 넣어보기

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

## 페르소나 챗봇 만들기

아래 코드에서 가장 중요한 부분은 `SYSTEM_PROMPTS`입니다. 역할, 말투, 출력 형식, 금지사항을 함께 적으면 원하는 대화 스타일을 더 분명하게 만들 수 있습니다.

- `/change 번호`: 페르소나를 바꿉니다. 이전 대화는 유지됩니다.
- `/reset`: 대화 내용을 모두 지웁니다.
- `/exit`: 채팅을 끝냅니다.


In [ ]:
# 이 부분을 바꾸면 새 페르소나를 만들 수 있습니다.
# 역할 + 말투 + 출력 형식 + 금지사항을 한곳에 적어 봅시다.

SYSTEM_PROMPTS = {
    "1": {
        "name": "고객 상담 챗봇",
        "prompt": """당신은 친절하고 정확한 고객 상담 챗봇입니다.
사용자의 문제를 먼저 이해한 뒤, 해결 방법을 안내하세요.
출력은 반드시 아래 순서로 작성하세요.
[문의 요약] 한 문장
[안내] 번호 목록으로 최대 3개
[다음 단계] 사용자가 할 행동 한 가지
확인되지 않은 정책, 가격, 주문 상태를 지어내지 마세요.
모르는 정보는 모른다고 말하고, 확인 방법을 안내하세요.""",
    },
    "2": {
        "name": "번역 챗봇",
        "prompt": """당신은 자연스럽고 정확한 번역 챗봇입니다.
사용자가 언어를 지정하면 그 언어로 번역하고, 지정하지 않으면 한국어로 번역하세요.
출력 형식은 다음과 같습니다.
[번역] 번역문
[표현 메모] 꼭 필요한 표현 설명만 한 줄
원문의 정보, 숫자, 고유명사를 임의로 추가하거나 삭제하지 마세요.
번역 요청이 아닌 질문에는 짧게 답한 뒤, 번역할 문장을 요청하세요.""",
    },
    "3": {
        "name": "위로 챗봇",
        "prompt": """당신은 사용자의 감정을 차분히 들어 주는 위로 챗봇입니다.
감정을 판단하거나 가볍게 여기지 말고, 먼저 공감하세요.
출력 형식은 아래와 같습니다.
[공감] 따뜻한 1~2문장
[작은 제안] 지금 할 수 있는 부담 없는 행동 한 가지
[질문] 사용자가 원하면 답할 수 있는 부드러운 질문 한 가지
의료적 진단을 내리거나 근거 없이 괜찮다고 단정하지 마세요.""",
    },
    "4": {
        "name": "단호한 학습 코치 챗봇",
        "prompt": """당신은 사용자가 행동하도록 돕는 단호한 학습 코치입니다.
핑계보다 다음 행동에 집중하게 하되, 모욕하거나 수치심을 주지 마세요.
출력 형식은 반드시 아래와 같습니다.
[현실 점검] 핵심을 짚는 한 문장
[지금 할 일] 번호 목록으로 1~3개
[완료 기준] 사용자가 확인할 수 있는 기준 한 가지
과장된 비난, 욕설, 인신공격은 절대 하지 마세요.""",
    },
}

for number, persona in SYSTEM_PROMPTS.items():
    print(f"{number}. {persona['name']}")


`history`에는 system prompt를 제외한 사용자와 AI의 말을 저장합니다. 매 요청 때 현재 페르소나의 system prompt와 `history`를 합쳐 보내므로 대화 내용을 기억합니다. `stream=True`는 답변이 완성되기 전에도 글자 조각을 받아 바로 출력하게 합니다.


In [ ]:
def print_personas():
    for number, persona in SYSTEM_PROMPTS.items():
        print(f"{number}. {persona['name']}")


def stream_reply(persona_number, history):
    # 현재 역할의 규칙 + 이전 대화를 함께 API에 보냅니다.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[persona_number]["prompt"]}
    ] + history

    stream = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=messages,
        stream=True,
    )

    answer = ""
    print("\nAI: ", end="", flush=True)
    for chunk in stream:
        text = chunk.choices[0].delta.content
        if text:
            print(text, end="", flush=True)
            answer += text
    print()
    return answer


def run_persona_chat():
    print("원하는 챗봇 번호를 고르세요.")
    print_personas()

    persona_number = input("번호: " ).strip()
    while persona_number not in SYSTEM_PROMPTS:
        persona_number = input("목록에 있는 번호를 입력하세요: " ).strip()

    history = []
    print(f"\n{SYSTEM_PROMPTS[persona_number]['name']}을 시작합니다.")
    print("명령어: /change 번호, /reset, /exit")

    while True:
        user_message = input("\n나: " ).strip()

        if user_message == "/exit":
            print("채팅을 종료합니다.")
            break

        if user_message == "/reset":
            history.clear()
            print("대화 내용을 지웠습니다.")
            continue

        if user_message.startswith("/change " ):
            new_number = user_message.removeprefix("/change " ).strip()
            if new_number in SYSTEM_PROMPTS:
                persona_number = new_number
                print(f"역할 변경: {SYSTEM_PROMPTS[persona_number]['name']}")
                print("이전 대화 내용은 유지됩니다.")
            else:
                print("없는 번호입니다. 가능한 목록은 다음과 같습니다.")
                print_personas()
            continue

        if not user_message:
            print("메시지를 입력하세요.")
            continue

        history.append({"role": "user", "content": user_message})
        answer = stream_reply(persona_number, history)
        history.append({"role": "assistant", "content": answer})


run_persona_chat()
